# Guardrail
Architecture:

```
                User
                  │
                  ▼
        +-------------------+
        | Guardrail LLM      |
        | (Classifier)       |
        +-------------------+
          YES         NO
           │           │
           ▼           ▼
   Main Assistant   Reject Request
           │
           ▼
       Response

In [1]:
import importlib.metadata

# List the distribution package names
packages = ["openai"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


openai version: 2.36.0


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# print(OPENAI_API_KEY)

# First Guardrail - Travel related classifier

In [15]:
import json

MODEL="gpt-4o-mini"

import json
from openai import OpenAI

MODEL = "gpt-4o-mini"

def is_travel_related(user_query: str) -> bool:
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    # 1. Define the exact JSON schema you want the model to follow
    json_schema = {
        "name": "travel_classifier",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "travel_related": {
                    "type": "boolean",
                    "description": "Whether the user query is related to travel, tourism, or geography."
                }
            },
            "required": ["travel_related"],
            "additionalProperties": False
        }
    }
    
    # 2. Use client.chat.completions.create (standard OpenAI SDK method)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a classifier. Determine if the query is travel related."
            },
            {
                "role": "user",
                "content": user_query
            }
        ],
        # Enforce the strict JSON schema
        response_format={"type": "json_schema", "json_schema": json_schema}
    )

    # 3. Extract the text message securely
    raw_content = response.choices[0].message.content
    
    try:
        result = json.loads(raw_content)
        # Use .get() with a default fallback to prevent KeyErrors
        return result.get("travel_related", False)
    except (json.JSONDecodeError, TypeError):
        # Fallback if JSON parsing somehow fails
        return False
    
# Test
print(is_travel_related(user_query="Which are popular landmarks in Paris ?"))  # Expected: True
print(is_travel_related(user_query="what is number theory ?"))                # Expected: False


True
False


# Second Guardrail - Only answer travel related queries

In [17]:
import os
from openai import OpenAI

def run_chat_session(system_prompt: str, model_name: str = MODEL):
    """Maintains a continuous conversation session with an OpenAI model."""
    
    # 1. Initialise the client and the conversation history
    client = OpenAI(api_key=OPENAI_API_KEY)
    conversation_history = [{"role": "system", "content": system_prompt}]
    
    print(f"Chat session started with model: {model_name}")
    print("Type 'exit' or 'q' to end the conversation.\n")
    
    # 2. Continuous user input loop
    while True:
        user_input = input("You: ").strip()
        
        # Check for exit commands or empty input
        if user_input.lower() in ['exit', 'quit', 'q']:
            print("Ending chat session.")
            break
        if not user_input:
            continue
            
        # 3. Append the new user question to the history
        conversation_history.append({"role": "user", "content": user_input})
        
        try:
            if not is_travel_related(user_query=user_input):
                print("Please ask ONLY travel related question")
                continue
                                 
            # 4. Send the entire chat history to OpenAI using standard syntax
            response = client.chat.completions.create(
                model=model_name,
                messages=conversation_history
            )

            
            # 5. Extract the assistant's response text correctly
            assistant_response = response.choices[0].message.content
            print(f"\nAI: {assistant_response}\n")
            
            # 6. Append the AI's response so it remembers it next turn
            conversation_history.append({"role": "assistant", "content": assistant_response})
            
        except Exception as e:
            print(f"\nAn error occurred: {e}\n")
            # Remove the last user message if the request failed to keep history clean
            conversation_history.pop()

# ==========================================
# Run the session
# ==========================================
SYSTEM_PROMPT = """
You are a travel assistant. 
Only answer questions related to travel. 
If the user asks about anything else, politely say:
"I'm only able to answer travel-related questions."
"""

run_chat_session(system_prompt=SYSTEM_PROMPT)

# What is 4 * 8
# what are major tourist spots in New Delhi

Chat session started with model: gpt-4o-mini
Type 'exit' or 'q' to end the conversation.



You:  What is 4 * 8


Please ask ONLY travel related question


You:  what are major tourist spots in New Delhi



AI: New Delhi is home to many major tourist spots, including:

1. **India Gate** - A prominent war memorial dedicated to Indian soldiers.
2. **Qutub Minar** - A UNESCO World Heritage Site and the tallest minaret in India.
3. **Lotus Temple** - A beautiful Bahá'í House of Worship known for its flower-like structure.
4. **Red Fort** - A historical fort and another UNESCO World Heritage Site, known for its stunning Mughal architecture.
5. **Humayun's Tomb** - A magnificent garden tomb, also a UNESCO World Heritage Site.
6. **Jama Masjid** - One of the largest mosques in India, notable for its impressive architecture.
7. **Akshardham Temple** - A modern Hindu temple complex known for its stunning artistry and cultural exhibitions.
8. **National Museum** - Displays a vast range of artifacts representing Indian history and culture.

These sites are just a starting point for exploring the rich history and culture of New Delhi!



You:  q


Ending chat session.
